### 1. 경로 설정

In [40]:
import sys
from pathlib import Path

# ✅ 노트북 위치가 어디든 프로젝트 루트를 찾아 sys.path에 추가
ROOT = Path.cwd()
while (ROOT / "src").exists() is False and ROOT.parent != ROOT:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT =", ROOT)

ROOT = c:\Users\최무영\Desktop\코드잇 스프린트\Team2-RAG-Project


### 2. 평가 데이터셋 로드

In [41]:
import json

DATA_PATH = ROOT / "evaluation" / "test_questions.json"  # 본인 파일명으로
with open(DATA_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

len(dataset), dataset[0].keys()

(30, dict_keys(['id', 'question', 'answer', 'evidence']))

### 3. 평가용 retriever 함수

In [48]:
from src.retriever import rerank_docs

def retrieve_docs(question: str, k: int):
    # candidate_k는 rerank에 넣을 후보 수 (top_n보다 넉넉히)
    candidate_k = max(60, k * 3)
    return rerank_docs(
        question,
        top_n=k,                 # ✅ K개 반환
        candidate_k=candidate_k, # ✅ 후보는 넉넉히
        docid_scope_top_n=None,  # ✅ 평가에서는 필터 OFF
    )

### 4. Recall, MRR 계산 함수

In [49]:
import os, re, unicodedata
from typing import Any, List, Dict, Optional, Tuple

def norm_fname(s: str) -> str:
    """
    - 유니코드 정규화(NFC): NFD/NFC 혼재 시 매칭 실패 방지
    - basename
    - lower
    - 공백 정리
    """
    if not s:
        return ""
    s = unicodedata.normalize("NFC", str(s))
    s = os.path.basename(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s.lower()

def doc_fname(doc) -> str:
    md = getattr(doc, "metadata", None) or {}
    fn = md.get("file_name") or md.get("filename") or md.get("source") or md.get("path") or ""
    return norm_fname(fn)

def first_rank_multi(docs: List[Any], gold_files: List[str], k: int) -> Optional[int]:
    """
    상위 k개 docs에서 gold_files 중 하나라도 최초 등장하는 rank(1-based)
    """
    gold_set = set(gold_files)
    for i, d in enumerate((docs or [])[:k], 1):
        if doc_fname(d) in gold_set:
            return i
    return None

def recall_mrr_at_k_multi(docs: List[Any], gold_files: List[str], k: int) -> Tuple[float, float]:
    r = first_rank_multi(docs, gold_files, k)
    if r is None:
        return 0.0, 0.0
    return 1.0, 1.0 / r

def gold_files_from_item(item: Dict[str, Any]) -> List[str]:
    """
    evidence 배열에서 source_file들을 뽑아 정규화한 리스트로 반환
    """
    evs = item.get("evidence") or []
    files = []
    for ev in evs:
        f = ev.get("source_file")
        if f:
            files.append(norm_fname(f))
    # 중복 제거(순서 유지)
    out = []
    for f in files:
        if f not in out:
            out.append(f)
    return out

In [50]:
from typing import Callable

def evaluate_retrieval(
    dataset: List[Dict[str, Any]],
    retrieve_fn: Callable[[str, int], List[Any]],
    k_list: List[int] = [1, 3, 5, 10],
    debug_n: int = 0,   # 처음 N개만 검색 결과 출력 (0이면 미출력)
):
    n = len(dataset)
    recall_sum = {k: 0.0 for k in k_list}
    mrr_sum = {k: 0.0 for k in k_list}

    for idx, item in enumerate(dataset, 1):
        q = item["question"]
        gold_doc = item["document"]

        # ✅ 가장 큰 K로 한 번만 가져오고, 슬라이싱으로 재사용 (속도 개선)
        max_k = max(k_list)
        docs = retrieve_fn(q, max_k)

        for k in k_list:
            r, m = recall_mrr_at_k(docs, gold_doc, k)
            recall_sum[k] += r
            mrr_sum[k] += m

        if debug_n and idx <= debug_n:
            print(f"\n[DEBUG Q{idx:02d}] {q}")
            print("GOLD_DOC:", gold_doc)
            print("TOP docs:")
            for j, d in enumerate(docs[:min(5, len(docs))], 1):
                print(f"  {j}. {_doc_fname(d)}")

    results = {
        "n": n,
        "Recall": {k: recall_sum[k] / n for k in k_list},
        "MRR": {k: mrr_sum[k] / n for k in k_list},
    }
    return results

### 5. 평가

In [ ]:
import pandas as pd

k_list = [1, 3, 5, 10, 20]  # 보고 싶은 K로 조정 (단, rerank_docs top_n이 이만큼 나와야 함)

results = evaluate_retrieval(
    dataset=dataset,
    retrieve_fn=retrieve_docs,
    k_list=k_list,
    debug_n=2,  # 처음 2개만 디버그 출력
)

df = pd.DataFrame({
    "K": list(results["Recall"].keys()),
    "Recall@K": list(results["Recall"].values()),
    "MRR@K": list(results["MRR"].values()),
})
df

### 6. BM25 / Dense(MMR) / Hybrid(RRF) / Rerank(FlashRank) 비교 평가

In [60]:
import sys, importlib
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.retriever as retriever
importlib.reload(retriever)
print("retriever file:", retriever.__file__)

retriever file: c:\Users\최무영\Desktop\코드잇 스프린트\Team2-RAG-Project\src\retriever.py


In [68]:
import os, re, unicodedata
from typing import Any, List, Dict, Optional, Tuple, Callable
import pandas as pd

# def norm_fname(s: str) -> str:
#     """
#     - 유니코드 정규화(NFC): NFD/NFC 혼재 시 매칭 실패 방지
#     - basename
#     - lower
#     - 공백 정리
#     """
#     if not s:
#         return ""
#     s = unicodedata.normalize("NFC", str(s))
#     s = os.path.basename(s).strip()
#     s = re.sub(r"\s+", " ", s)
#     return s.lower()

# def doc_fname(doc) -> str:
#     md = getattr(doc, "metadata", None) or {}
#     fn = md.get("file_name") or md.get("filename") or md.get("source") or md.get("path") or ""
#     return norm_fname(fn)

# def gold_files_from_item(item: Dict[str, Any]) -> List[str]:
#     """
#     evidence 배열에서 source_file들을 뽑아 정규화한 리스트로 반환
#     """
#     evs = item.get("evidence") or []
#     files = []
#     for ev in evs:
#         f = ev.get("source_file")
#         if f:
#             files.append(norm_fname(f))
#     # 중복 제거(순서 유지)
#     out = []
#     for f in files:
#         if f not in out:
#             out.append(f)
#     return out

def norm_doc_id(s: str) -> str:
    if not s:
        return ""
    s = unicodedata.normalize("NFC", str(s))
    s = os.path.basename(s).strip()
    s = re.sub(r"\.[A-Za-z0-9]+$", "", s)  # ✅ 확장자 제거
    s = re.sub(r"\s+", " ", s)
    return s.lower()

def gold_doc_ids_from_item(item):
    evs = item.get("evidence") or []
    out = []
    for ev in evs:
        f = ev.get("source_file")
        if f:
            did = norm_doc_id(f)
            if did not in out:
                out.append(did)
    return out

def doc_id_from_doc(doc):
    md = getattr(doc, "metadata", None) or {}
    fn = md.get("file_name") or md.get("filename") or md.get("source") or md.get("path") or ""
    return norm_doc_id(fn)


def first_rank_multi(docs: List[Any], gold_files: List[str], k: int) -> Optional[int]:
    """
    상위 k개 docs에서 gold_files 중 하나라도 최초 등장하는 rank(1-based)
    """
    gold_set = set(gold_files)
    for i, d in enumerate((docs or [])[:k], 1):
        if doc_id_from_doc(d) in gold_set:
            return i
    return None

def recall_mrr_at_k_multi(docs: List[Any], gold_files: List[str], k: int) -> Tuple[float, float]:
    r = first_rank_multi(docs, gold_files, k)
    if r is None:
        return 0.0, 0.0
    return 1.0, 1.0 / r

def evaluate_retrieval(
    dataset: List[Dict[str, Any]],
    retrieve_fn: Callable[[str, int], List[Any]],
    k_list: List[int] = [1, 3, 5, 10, 20],
    debug_n: int = 0,
    label: str = "",
):
    n = len(dataset)
    max_k = max(k_list)

    recall_sum = {k: 0.0 for k in k_list}
    mrr_sum = {k: 0.0 for k in k_list}

    for idx, item in enumerate(dataset, 1):
        q = item["question"]
        gold_files = gold_doc_ids_from_item(item)  # ✅ 이름 맞추기

        docs = retrieve_fn(q, max_k) or []

        for k in k_list:
            r, m = recall_mrr_at_k_multi(docs, gold_files, k)
            recall_sum[k] += r
            mrr_sum[k] += m

        if debug_n and idx <= debug_n:
            print(f"\n[DEBUG:{label} Q{idx:02d}] {q}")
            print("GOLD_DOC_IDS:")
            for gf in gold_files:
                print(" -", gf)
            print("TOP5:")
            for j, d in enumerate(docs[:5], 1):
                print(f"  {j}. {doc_id_from_doc(d)}")  # ✅ 평가 기준과 동일하게 출력

    return {
        "n": n,
        "Recall": {k: recall_sum[k] / n for k in k_list},
        "MRR": {k: mrr_sum[k] / n for k in k_list},
    }

### 7. 단계 별 함수

- BM25

In [69]:
def retrieve_bm25(question: str, k: int):
    r = retriever.get_bm25_retriever()
    docs = r.invoke(question) or []
    return docs[:k]

- Dense

In [70]:
def retrieve_dense(question: str, k: int):
    r = retriever.get_dense_retriever()
    docs = r.invoke(question) or []
    return docs[:k]

- Hybrid

In [71]:
import inspect

_hsig = inspect.signature(retriever.get_hybrid_docs)
_hparams = _hsig.parameters

def retrieve_hybrid(question: str, k: int):
    candidate_k = max(60, k * 3)

    # 신버전(get_hybrid_docs(question, k=...))이면 candidate_k를 넣기
    if "k" in _hparams:
        docs = retriever.get_hybrid_docs(question, k=candidate_k) or []
        return docs[:k]

    # 구버전이면 그냥 받아서 slice (구버전은 내부 k=60 고정일 확률 높음)
    docs = retriever.get_hybrid_docs(question) or []
    return docs[:k]

- Rerank

In [72]:
_rsig = inspect.signature(retriever.rerank_docs)
_rparams = _rsig.parameters

def retrieve_rerank(question: str, k: int):
    if "top_n" in _rparams:
        candidate_k = max(60, k * 3) if "candidate_k" in _rparams else None
        kwargs = {"top_n": k}
        if candidate_k is not None:
            kwargs["candidate_k"] = candidate_k
        if "docid_scope_top_n" in _rparams:
            kwargs["docid_scope_top_n"] = None  # 평가용: 필터 OFF
        return retriever.rerank_docs(question, **kwargs) or []
    docs = retriever.rerank_docs(question) or []
    return docs[:k]

### 8. 평가

In [73]:
k_list = [1, 3, 5, 10, 20]
runs = [
    ("BM25",   retrieve_bm25),
    ("Dense",  retrieve_dense),
    ("Hybrid", retrieve_hybrid),
    ("Rerank", retrieve_rerank),
]

rows = []
for name, fn in runs:
    res = evaluate_retrieval(dataset, fn, k_list=k_list, debug_n=3, label=name)
    for k in k_list:
        rows.append({"Stage": name, "K": k, "Recall@K": res["Recall"][k], "MRR@K": res["MRR"][k]})

df = pd.DataFrame(rows).sort_values(["K", "Stage"])
df


[DEBUG:BM25 Q01] 봉화군에서 진행하는 재난 시스템 고도화 사업 예산이 총 얼마로 잡혀 있나요? 주관 부서가 어디인지도 확인 부탁드립니다.
GOLD_DOC_IDS:
 - 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
TOP5:
  1. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  2. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  3. 국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축
  4. 서영대학교 산학협력단_전문대학 혁신지원사업 서영대학교 차세대 교육
  5. 국방과학연구소_대용량 자료전송시스템 고도화

[DEBUG:BM25 Q02] 이번 봉화군 재난위험지역 경보발령 사업에서 활용하는 무선 통신 방식은 구체적으로 무엇인가요?
GOLD_DOC_IDS:
 - 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
TOP5:
  1. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  2. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  3. 경기도 평택시_2024년도 평택시 버스정보시스템(bis) 구축사업
  4. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  5. 대검찰청_아태 사이버범죄 역량강화 허브(apc-hub) 홈페이지 및 온라인 교

[DEBUG:BM25 Q03] 봉화군 재난 사업용 하드웨어 규격 중에서 ECR-02 무선경보장치 스피커의 출력 기준을 알려주세요.
GOLD_DOC_IDS:
 - 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
TOP5:
  1. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  2. 국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축
  3. 국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축
  4. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



[DEBUG:Dense Q01] 봉화군에서 진행하는 재난 시스템 고도화 사업 예산이 총 얼마로 잡혀 있나요? 주관 부서가 어디인지도 확인 부탁드립니다.
GOLD_DOC_IDS:
 - 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
TOP5:
  1. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  2. 국가철도공단_철도인프라 디지털트윈 정보화전략계획(isp) 수립 용역(변
  3. 광주과학기술원_실시간통합연구비관리시스템(rcms) 연계 모듈 변경 사업
  4. 한국사학진흥재단_대학재정정보시스템(기본재산 및 기채 사후관리) 고
  5. 인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 

[DEBUG:Dense Q02] 이번 봉화군 재난위험지역 경보발령 사업에서 활용하는 무선 통신 방식은 구체적으로 무엇인가요?
GOLD_DOC_IDS:
 - 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
TOP5:
  1. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  2. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  3. (사）한국대학스포츠협의회_kusf 체육특기자 경기기록 관리시스템 개발
  4. koica 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 
  5. 국가철도공단_철도인프라 디지털트윈 정보화전략계획(isp) 수립 용역(변


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



[DEBUG:Dense Q03] 봉화군 재난 사업용 하드웨어 규격 중에서 ECR-02 무선경보장치 스피커의 출력 기준을 알려주세요.
GOLD_DOC_IDS:
 - 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
TOP5:
  1. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  2. 한국수자원조사기술원_수문자료정보관리시스템(hdims) 재구축 용역(3단계
  3. 서울특별시 여성가족재단_(재공고, 협상) 서울 디지털성범죄 안심지원센
  4. 인천공항운영서비스(주)_인천공항운영서비스㈜ 차세대 erp시스템 구축 
  5. 축산물품질평가원_축산물이력관리시스템 개선(정보화 사업)


...


[DEBUG:Rerank Q01] 봉화군에서 진행하는 재난 시스템 고도화 사업 예산이 총 얼마로 잡혀 있나요? 주관 부서가 어디인지도 확인 부탁드립니다.
GOLD_DOC_IDS:
 - 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
TOP5:
  1. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  2. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  3. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  4. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  5. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



[DEBUG:Rerank Q02] 이번 봉화군 재난위험지역 경보발령 사업에서 활용하는 무선 통신 방식은 구체적으로 무엇인가요?
GOLD_DOC_IDS:
 - 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
TOP5:
  1. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  2. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  3. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  4. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  5. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)

[DEBUG:Rerank Q03] 봉화군 재난 사업용 하드웨어 규격 중에서 ECR-02 무선경보장치 스피커의 출력 기준을 알려주세요.
GOLD_DOC_IDS:
 - 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
TOP5:
  1. 파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업
  2. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  3. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  4. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)
  5. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embedding

,Stage,K,Recall@K,MRR@K
0,BM25,1,0.500000,0.500000
5,Dense,1,0.533333,0.533333
10,Hybrid,1,0.566667,0.566667
15,Rerank,1,0.433333,0.433333
1,BM25,3,0.566667,0.533333
6,Dense,3,0.733333,0.622222
11,Hybrid,3,0.700000,0.627778
16,Rerank,3,0.633333,0.527778
2,BM25,5,0.600000,0.541667
7,Dense,5,0.733333,0.622222


In [74]:
pivot_recall = df.pivot(index="K", columns="Stage", values="Recall@K")
pivot_mrr    = df.pivot(index="K", columns="Stage", values="MRR@K")

**Recall**

In [75]:
pivot_recall

Stage,BM25,Dense,Hybrid,Rerank
K,,,,
1,0.500000,0.533333,0.566667,0.433333
3,0.566667,0.733333,0.700000,0.633333
5,0.600000,0.733333,0.866667,0.766667
10,0.700000,0.833333,0.900000,0.800000
20,0.766667,0.900000,0.933333,0.933333


**MRR**

In [76]:
pivot_mrr

Stage,BM25,Dense,Hybrid,Rerank
K,,,,
1,0.500000,0.533333,0.566667,0.433333
3,0.533333,0.622222,0.627778,0.527778
5,0.541667,0.622222,0.664444,0.557778
10,0.553704,0.636706,0.668611,0.561111
20,0.557934,0.640750,0.670278,0.570438
